In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

Load and preprocess and shuffle the dataframe

In [ ]:
df = pd.read_csv('data/final.csv')

df['key'] = df['key'].astype('category')

df['mode'] = df['mode'].astype('category')


Shuffle data

In [ ]:
df = df.sample(frac=1, random_state=42)


Extract song name

In [ ]:
song_names = df['name']


Remove extra columns (by keeping the ones we need)

In [ ]:

df = df[['danceability', 'energy','key','loudness','mode','speechiness','acousticness','instrumentalness','liveness','valence','tempo','preference']]



Remove duplicates

In [ ]:
df = df.drop_duplicates()


Split into feature and target variable, although for this analysis we will not use our target variable.

In [ ]:
X = df.iloc[:, 1:-1].values
y = df.iloc[:, -1].values


Define a scaling function

In [ ]:
def scale_features(X):
    for i in range(X.shape[1]):
        col = X[:, i]
        if np.issubdtype(col.dtype, np.number):  # check if the column is numeric
            col = col.astype(float)
            X[:, i] = (col - col.mean()) / col.std()  # scale the column


Apply scaling function

In [ ]:
scale_features(X)


Define the number of clusters we want

In [ ]:
k = 2


Initizlialize the KMeans object

In [ ]:
kmeans = KMeans(n_clusters=k)


Apply the function to our feature variables

In [ ]:
kmeans.fit(X)


Get the cluster labels for each data point

In [ ]:
labels = kmeans.labels_


Add the cluter labels to the original data, as a new column

In [ ]:
df['cluster'] = labels


Let's see how many data points are in each cluster

In [ ]:
print(df['cluster'].value_counts())


Group the data by cluster

In [ ]:
clusters = df.groupby('cluster')


Iterate over the cluters and print the details

In [ ]:
for cluster, cluster_data in clusters:
    print('Cluster {}:'.format(cluster))
    print('Number of data points: {}'.format(len(cluster_data)))
    print('Mean values of features:')
    print(cluster_data.mean())
    print('\n')


Perform PCA to project the data onto two principal components (the same thing we did in the PCA analysis)

In [ ]:
pca = PCA(n_components=2)
pca_data = pca.fit_transform(X)


# define a dictionary to map clusters to colors
colors = {0: 'teal', 1: 'purple'}

# define marker size
marker_size = 2


plt.scatter(pca_data[:,0], pca_data[:,1], c=df['cluster'].map(colors), s=50, alpha=0.6, edgecolors='w')
# add legends
legend_elements = [plt.Line2D([0], [0], marker='o', color=color, label='Cluster {}'.format(cluster))
                   for cluster, color in colors.items()]
plt.legend(handles=legend_elements)

plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('K-Means Clustering Results')
plt.show()

In [ ]:
# plt.savefig('kmeanscluter.png')